In [1]:
import numpy as np

def compute_mae(var, var_hat):
    return np.sum(np.abs(var - var_hat)) / var.shape[0]

def compute_rmse(var, var_hat):
    return np.sqrt(np.sum((var - var_hat) ** 2) / var.shape[0])

In [2]:
import cvxpy as cp
import numpy as np

def huber_ar(y, d, delta = 1, solver = cp.ECOS):
    t = y.shape[0]
    vec = y[d :] # T-d
    mat = np.vstack([y[d-i-1 : t-i-1] for i in range(d)]).T # (T-d) x d

    coef = cp.Variable(d)
    residual = vec - mat @ coef
    obj = cp.sum(cp.huber(residual, delta))
    prob = cp.Problem(cp.Minimize(obj))
    prob.solve(solver = solver)

    return coef.value

def time_series_pred_multi(data, coef, horizon, step):
    t = data.shape[0]
    d = coef.shape[0]

    origins = np.arange(t - horizon, t) - step + 1             # (horizon,)
    assert origins.min() - d >= 0, "history too short"

    win = np.stack([data[o - d : o] for o in origins])         # horizon x d

    for _ in range(step):
        pred = win[:, ::-1] @ coef                             # (horizon,)
        win = np.hstack([win[:, 1:], pred[:, None]])

    return pred

(CVXPY) Oct 09 05:38:40 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Oct 09 05:38:40 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')


In [3]:
import pandas as pd

df = pd.read_csv('AI-small-data.csv', index_col = 0)

In [4]:
x = df.loc['Machine_learning'].values[: 24 * 31]

d = 168
for delta in range(1, 50, 1):
    coef = huber_ar(x[: 24 * (31 - 14)], d, delta)
    horizon = 168
    step = 12
    data = x[: - 24 * 7]
    est = time_series_pred_multi(x, coef, horizon, step)

    print('Huber threshold: {}'.format(delta))
    print('MAE: {:.6}'.format(compute_mae((data[- horizon :]).reshape(-1), est.reshape(-1))))
    print('RMSE: {:.6}'.format(compute_rmse((data[- horizon :]).reshape(-1), est.reshape(-1))))
    print()

/opt/anaconda3/lib/python3.11/site-packages/cvxpy/reductions/solvers/solving_chain.py:356: FutureWarning: 
    You specified your problem should be solved by ECOS. Starting in
    CXVPY 1.6.0, ECOS will no longer be installed by default with CVXPY.
    Please either add ECOS as an explicit install dependency to your project
    or switch to our new default solver, Clarabel, by either not specifying a
    solver argument or specifying ``solver=cp.CLARABEL``. To suppress this
    warning while continuing to use ECOS, you can filter this warning using
    Python's ``warnings`` module until you are using 1.6.0.
    
  warnings.warn(ECOS_DEP_DEPRECATION_MSG, FutureWarning)


Huber threshold: 1
MAE: 68.4838
RMSE: 90.4322

Huber threshold: 2
MAE: 67.4414
RMSE: 88.8551

Huber threshold: 3
MAE: 65.6958
RMSE: 86.8592

Huber threshold: 4
MAE: 64.3965
RMSE: 85.439

Huber threshold: 5
MAE: 63.6651
RMSE: 84.7917

Huber threshold: 6
MAE: 63.9294
RMSE: 84.8887

Huber threshold: 7
MAE: 64.2171
RMSE: 85.2483

Huber threshold: 8
MAE: 64.3533
RMSE: 85.4085

Huber threshold: 9
MAE: 64.2555
RMSE: 85.3897

Huber threshold: 10
MAE: 63.9391
RMSE: 85.1175

Huber threshold: 11
MAE: 63.6522
RMSE: 84.9161

Huber threshold: 12
MAE: 63.4188
RMSE: 84.818

Huber threshold: 13
MAE: 63.1288
RMSE: 84.6292

Huber threshold: 14
MAE: 62.7914
RMSE: 84.3364

Huber threshold: 15
MAE: 62.6113
RMSE: 84.0963

Huber threshold: 16
MAE: 62.4962
RMSE: 83.9239

Huber threshold: 17
MAE: 62.4966
RMSE: 83.7925

Huber threshold: 18
MAE: 62.5296
RMSE: 83.7046

Huber threshold: 19
MAE: 62.4369
RMSE: 83.513

Huber threshold: 20
MAE: 62.4674
RMSE: 83.5014

Huber threshold: 21
MAE: 62.5818
RMSE: 83.585

Huber

In [5]:
d = 168
delta = 20
coef = huber_ar(x[: 24 * (31 - 14)], d, delta)
horizon = 168
est = time_series_pred_multi(x, coef, horizon, step)

print('Huber threshold: {}'.format(delta))
print('MAE: {:.6}'.format(compute_mae((x[- horizon :]).reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse((x[- horizon :]).reshape(-1), est.reshape(-1))))
print()

Huber threshold: 20
MAE: 57.9158
RMSE: 73.4503



In [6]:
mat = np.zeros((horizon, 2))
mat[:, 0] = np.arange(1, 169)
mat[:, 1] = est
np.savetxt('ml_result.txt', mat)

In [7]:
x = df.loc['Bayesian_network'].values[: 24 * 31]

d = 168
for delta in range(1, 50, 1):
    coef = huber_ar(x[: 24 * (31 - 14)], d, delta)
    horizon = 168
    step = 12
    data = x[: - 24 * 7]
    est = time_series_pred_multi(x, coef, horizon, step)

    print('Huber threshold: {}'.format(delta))
    print('MAE: {:.6}'.format(compute_mae((data[- horizon :]).reshape(-1), est.reshape(-1))))
    print('RMSE: {:.6}'.format(compute_rmse((data[- horizon :]).reshape(-1), est.reshape(-1))))
    print()

Huber threshold: 1
MAE: 7.65743
RMSE: 9.47031

Huber threshold: 2
MAE: 7.18694
RMSE: 8.93804

Huber threshold: 3
MAE: 6.72236
RMSE: 8.39581

Huber threshold: 4
MAE: 6.31248
RMSE: 7.97159

Huber threshold: 5
MAE: 6.03727
RMSE: 7.68268

Huber threshold: 6
MAE: 5.94786
RMSE: 7.55852

Huber threshold: 7
MAE: 5.91842
RMSE: 7.50917

Huber threshold: 8
MAE: 5.9117
RMSE: 7.50516

Huber threshold: 9
MAE: 5.92779
RMSE: 7.53446

Huber threshold: 10
MAE: 5.9667
RMSE: 7.58256

Huber threshold: 11
MAE: 6.01179
RMSE: 7.62438

Huber threshold: 12
MAE: 6.03441
RMSE: 7.64087

Huber threshold: 13
MAE: 6.0434
RMSE: 7.64474

Huber threshold: 14
MAE: 6.05279
RMSE: 7.65467

Huber threshold: 15
MAE: 6.06471
RMSE: 7.67066

Huber threshold: 16
MAE: 6.08619
RMSE: 7.69272

Huber threshold: 17
MAE: 6.10787
RMSE: 7.71741

Huber threshold: 18
MAE: 6.10787
RMSE: 7.71742

Huber threshold: 19
MAE: 6.10787
RMSE: 7.71742

Huber threshold: 20
MAE: 6.10787
RMSE: 7.71742

Huber threshold: 21
MAE: 6.10787
RMSE: 7.71742

Hube

/opt/anaconda3/lib/python3.11/site-packages/cvxpy/problems/problem.py:1407: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


Huber threshold: 47
MAE: 6.10787
RMSE: 7.71742

Huber threshold: 48
MAE: 6.10787
RMSE: 7.71742

Huber threshold: 49
MAE: 6.10787
RMSE: 7.71742



In [8]:
d = 168
delta = 8
coef = huber_ar(x[: 24 * (31 - 14)], d, delta)
horizon = 168
est = time_series_pred_multi(x, coef, horizon, step)

print('Huber threshold: {}'.format(delta))
print('MAE: {:.6}'.format(compute_mae((x[- horizon :]).reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse((x[- horizon :]).reshape(-1), est.reshape(-1))))
print()

Huber threshold: 8
MAE: 6.11836
RMSE: 7.97046



In [9]:
mat = np.zeros((horizon, 2))
mat[:, 0] = np.arange(1, 169)
mat[:, 1] = est
np.savetxt('bn_result.txt', mat)